# 9주차: 로지스틱 회귀 I — 교실 실습
> 인공지능수학 | 교실 실습용 노트북  
> 구성: 손문제 (35분) → 🐛 버그 잡기 팀활동 (20분) → 코딩 실습 (20분)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

---
## Part 1. 손문제 (35분)

### 문제 1 — 시그모이드 유도 따라가기

$P(y=1 \mid x) = p$라 할 때, 아래 각 변환 단계를 채우시오.

| 단계 | 변환 | 범위 |
|---|---|---|
| 원래 확률 | $p$ | $[0, 1]$ |
| **Step 1** | Odds $= $ **(1)** | $[0, +\infty)$ |
| **Step 2** | logit $= $ **(2)** | $(-\infty, +\infty)$ |
| **Step 3** | logit $= wx + b$ 로 설정 | — |
| **Step 4** | $p = $ **(3)** (역으로 풀기) | $(0, 1)$ |

**(4)** Step 4의 역산 과정을 손으로 직접 유도하시오.  
($\log\frac{p}{1-p} = z$ 에서 시작하여 $p$에 대해 풀기)

---
*(풀이 공간)*






### 문제 2 — 시그모이드 계산과 성질 확인

**(1)** $\sigma(0)$, $\sigma(1)$, $\sigma(-1)$, $\sigma(2)$를 손으로 계산하시오.  
(계산기 없이, $e \approx 2.718$, $e^2 \approx 7.389$를 이용)

**(2)** $\sigma(1) + \sigma(-1)$을 계산하시오. 어떤 성질이 확인되는가?

**(3)** $w = 2, b = -1$인 로지스틱 회귀 모델에서 결정 경계의 $x$ 값을 구하시오.  
(결정 경계: $\hat{p} = 0.5$, 즉 $\sigma(z) = 0.5$인 $z$를 먼저 구한 후)

---
*(풀이 공간)*






### 문제 3 — BCE 손실 유도

3개의 훈련 데이터가 있습니다:

| $i$ | $y_i$ | $\hat{p}_i$ |
|:---:|:---:|:---:|
| 1 | 1 | 0.8 |
| 2 | 0 | 0.3 |
| 3 | 1 | 0.4 |

**(1)** 각 데이터 포인트의 $P(y_i \mid x_i)$를 계산하시오.  
(공식: $P(y \mid x) = \hat{p}^y (1-\hat{p})^{1-y}$)

**(2)** 전체 우도 $L = \prod_{i=1}^3 P(y_i \mid x_i)$를 계산하시오.

**(3)** $\ell = \log L$을 계산하시오. ($\log 0.8 \approx -0.223$, $\log 0.7 \approx -0.357$, $\log 0.4 \approx -0.916$)

**(4)** BCE $= -\ell / n$을 계산하시오.

**(5)** 만약 세 번째 데이터의 $\hat{p}_3$이 $0.4 \to 0.9$로 개선되면 BCE는 어떻게 변하는가?  
($\log 0.9 \approx -0.105$를 이용)

---
*(풀이 공간)*






### 문제 4 — MLE와 BCE의 동치 관계 서술

8주차에서 배운 MLE와 이번 주 BCE의 관계를 아래 빈칸을 채워 완성하시오.

이진 분류에서 각 데이터 포인트가 독립적으로 생성되었다고 가정하면, 전체 우도는:

$$L(w, b) = \prod_{i=1}^n \hat{p}_i^{\,(1)} \cdot (1-\hat{p}_i)^{\,(2)}$$

로그를 취하면:

$$\ell(w, b) = \sum_{i=1}^n \left[ (3) \cdot \log \hat{p}_i + (4) \cdot \log(1-\hat{p}_i) \right]$$

경사하강법은 최소화를 수행하므로 부호를 바꾸고 $n$으로 나누면:

$$\mathcal{L}_{\text{BCE}} = -\frac{1}{n} \cdot (5)$$

따라서 **BCE 최소화 = MLE (6)**.

---
*(풀이 공간)*





---
## Part 2. 🐛 버그 잡기 팀활동 (20분)

> 영상에서 본 버그입니다. 팀원들과 논의하여 원인을 찾고 수정하세요.

### 버그 설명

아래 코드는 로지스틱 회귀를 학습하는데, 손실이 `log(2) ≈ 0.693`에 머물며 전혀 줄어들지 않습니다.  
**버그 유형: 업데이트 방향 오류 (`-=` vs `+=`)**

In [ ]:
# 🐛 Buggy code
np.random.seed(1)
X_bug = np.concatenate([np.random.randn(40) - 2,
                         np.random.randn(40) + 2])
y_bug = np.array([0]*40 + [1]*40)

def bce_loss(y_true, y_pred):
    y_pred = np.clip(y_pred, 1e-12, 1 - 1e-12)
    return -(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred)).mean()

w_bug, b_bug = 0.0, 0.0
lr_bug = 0.1
losses_bug = []

for _ in range(150):
    z_bug  = w_bug * X_bug + b_bug
    p_bug  = sigmoid(z_bug)
    losses_bug.append(bce_loss(y_bug, p_bug))

    dw = ((p_bug - y_bug) * X_bug).mean()
    db = (p_bug - y_bug).mean()

    w_bug += lr_bug * dw   # <-- Bug!
    b_bug += lr_bug * db

plt.figure(figsize=(8, 3))
plt.plot(losses_bug)
plt.axhline(np.log(2), color='red', linestyle='--', label=f'log(2) = {np.log(2):.4f}')
plt.xlabel('Epoch')
plt.ylabel('BCE Loss')
plt.title('Buggy training')
plt.legend()
plt.tight_layout()
plt.show()

print(f"최종 w = {w_bug:.4f}, b = {b_bug:.4f}")

**팀 토론 가이드:**

1. `w ≈ 0`이고 손실이 `log(2)`에 머문다는 것은 모델이 항상 $\hat{p} \approx 0.5$를 예측한다는 의미입니다.  
   왜 그런지 설명하세요.

2. 경사하강법의 업데이트 규칙은 $w \leftarrow w - \alpha \cdot \frac{\partial L}{\partial w}$입니다.  
   코드의 `+=`이 이 규칙과 어떻게 다른가요?

3. `+=`를 사용하면 파라미터가 어느 방향으로 이동하고, 손실이 어떻게 변할지 예상하세요.

수정된 코드를 아래에 작성하세요:

In [ ]:
# ✅ Fixed code
w_fix, b_fix = 0.0, 0.0
losses_fix = []

for _ in range(150):
    z_fix  = w_fix * X_bug + b_fix
    p_fix  = sigmoid(z_fix)
    losses_fix.append(bce_loss(y_bug, p_fix))

    dw = ((p_fix - y_bug) * X_bug).mean()
    db = (p_fix - y_bug).mean()

    w_fix ______ lr_bug * dw   # 올바른 업데이트 방향으로 수정하세요
    b_fix ______ lr_bug * db

plt.figure(figsize=(8, 3))
plt.plot(losses_bug, 'r--', alpha=0.5, label='buggy')
plt.plot(losses_fix, 'b-',             label='fixed')
plt.xlabel('Epoch')
plt.ylabel('BCE Loss')
plt.title('Buggy vs Fixed training')
plt.legend()
plt.tight_layout()
plt.show()

print(f"수정 후 w = {w_fix:.3f}, b = {b_fix:.3f}")

---
## Part 3. 코딩 실습 (20분)

### 실습 1 — 손문제 3번 검증

In [ ]:
# Verify Problem 3 hand calculations
y_true = np.array([1, 0, 1])
p_hat  = np.array([0.8, 0.3, 0.4])

# (1) P(y_i | x_i) for each data point
prob_each = p_hat**y_true * (1 - p_hat)**(1 - y_true)

# (2) Total likelihood
L_total = np.prod(prob_each)

# (3) log-likelihood
log_L = np.log(L_total)

# (4) BCE
bce = -log_L / len(y_true)

print("각 데이터 포인트의 P(y|x):")
for i, (y, p, prob) in enumerate(zip(y_true, p_hat, prob_each)):
    print(f"  i={i+1}: y={y}, p_hat={p}, P(y|x)={prob:.4f}")
print(f"\n전체 우도 L     = {L_total:.6f}")
print(f"log-likelihood  = {log_L:.4f}")
print(f"BCE             = {bce:.4f}")
print()

# (5) Improved prediction
p_hat_improved = np.array([0.8, 0.3, 0.9])
bce_improved = -np.log(p_hat_improved**y_true * (1-p_hat_improved)**(1-y_true)).mean()
print(f"p3 개선 후 BCE = {bce_improved:.4f}  (이전: {bce:.4f})")
print(f"손실이 줄었는가? {bce_improved < bce}")

### 실습 2 — 로지스틱 회귀 완전 구현 및 결정 경계 시각화

In [ ]:
# Full logistic regression with decision boundary visualization
np.random.seed(42)
X_pos = np.random.randn(50) + 2.0
X_neg = np.random.randn(50) - 2.0
X_all = np.concatenate([X_neg, X_pos])
y_all = np.array([0]*50 + [1]*50)

# Train
w, b = 0.0, 0.0
lr, n = 0.3, len(y_all)
losses = []

for epoch in range(300):
    z     = w * X_all + b
    p_hat = sigmoid(z)
    losses.append(bce_loss(y_all, p_hat))
    w -= lr * ((p_hat - y_all) * X_all).mean()
    b -= lr * (p_hat - y_all).mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: loss curve
axes[0].plot(losses, 'steelblue')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('BCE Loss')
axes[0].set_title('Training Loss')

# Right: decision boundary
x_line  = np.linspace(-6, 6, 300)
p_line  = sigmoid(w * x_line + b)
boundary = -b / w

axes[1].scatter(X_neg, np.zeros(50), color='royalblue', label='class 0', alpha=0.6)
axes[1].scatter(X_pos, np.zeros(50), color='tomato',    label='class 1', alpha=0.6)
axes[1].plot(x_line, p_line, 'k-', linewidth=2, label='P(y=1|x)')
axes[1].axvline(boundary, color='green', linestyle='--', label=f'boundary x={boundary:.2f}')
axes[1].axhline(0.5, color='gray', linestyle=':', alpha=0.5)
axes[1].set_xlabel('x')
axes[1].set_ylabel('Probability')
axes[1].set_title('Learned Decision Boundary')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

# Accuracy
y_pred_label = (sigmoid(w * X_all + b) >= 0.5).astype(int)
accuracy = (y_pred_label == y_all).mean()
print(f"w = {w:.3f}, b = {b:.3f}")
print(f"결정 경계: x = {boundary:.3f}")
print(f"정확도: {accuracy*100:.1f}%")

### 실습 3 — BCE vs MSE 비교

같은 데이터로 BCE와 MSE로 각각 학습했을 때 손실 곡선을 비교하세요.

In [ ]:
# Compare BCE vs MSE for logistic regression training
def train_logistic(X, y, loss_type='bce', lr=0.3, epochs=300):
    w, b = 0.0, 0.0
    losses = []
    for _ in range(epochs):
        z     = w * X + b
        p_hat = sigmoid(z)

        if loss_type == 'bce':
            loss = bce_loss(y, p_hat)
            dw = ((p_hat - y) * X).mean()
            db = (p_hat - y).mean()
        else:  # MSE
            loss = ((p_hat - y)**2).mean()
            # d(MSE)/dw = (p_hat - y) * sigma'(z) * x
            d_sigma = p_hat * (1 - p_hat)   # sigma'(z)
            dw = ((p_hat - y) * d_sigma * X).mean()
            db = ((p_hat - y) * d_sigma).mean()

        losses.append(loss)
        w -= lr * dw
        b -= lr * db
    return losses, w, b

losses_bce, w_bce, b_bce = train_logistic(X_all, y_all, 'bce')
losses_mse, w_mse, b_mse = train_logistic(X_all, y_all, 'mse')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(losses_bce, 'royalblue', label='BCE')
axes[0].plot(losses_mse, 'tomato',    label='MSE')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (own scale)')
axes[0].set_title('Loss curves: BCE vs MSE')
axes[0].legend()

x_line = np.linspace(-6, 6, 300)
axes[1].plot(x_line, sigmoid(w_bce*x_line + b_bce), 'royalblue', linewidth=2, label=f'BCE  w={w_bce:.2f}')
axes[1].plot(x_line, sigmoid(w_mse*x_line + b_mse), 'tomato',    linewidth=2, label=f'MSE  w={w_mse:.2f}')
axes[1].axhline(0.5, color='gray', linestyle=':', alpha=0.5)
axes[1].scatter(X_neg[:20], np.zeros(20), color='royalblue', alpha=0.3, s=20)
axes[1].scatter(X_pos[:20], np.zeros(20), color='tomato',    alpha=0.3, s=20)
axes[1].set_xlabel('x')
axes[1].set_ylabel('P(y=1|x)')
axes[1].set_title('Learned sigmoid: BCE vs MSE')
axes[1].legend()

plt.tight_layout()
plt.show()

print("두 모델의 결정 경계:")
print(f"  BCE: x = {-b_bce/w_bce:.3f}")
print(f"  MSE: x = {-b_mse/w_mse:.3f}")